In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))


from kaggle_environments import make

from src.agents.chi8 import agent

In [ ]:
env = make(
    "kaggriculture",
    configuration={"episodeSteps": 720},
    debug=True,
)

In [ ]:
env.run([agent, "random"])

In [ ]:
final = env.steps[-1]

for i, state in enumerate(final):
    print(f"Player {i}: reward={state.reward}, status={state.status}")

In [ ]:
from IPython.display import HTML, display

html = env.render(mode="html", width=1200, height=800)
display(HTML(html))

## États des animaux et plantations


In [ ]:
obs = env.steps[23][0]["observation"]
farm = obs["farms"][obs["player"]]
for y, row in enumerate(farm["tiles"]):
    for x, tile in enumerate(row):
        if not isinstance(tile, dict):
            continue
        if tile["kind"] == "PLANT":
            print(f"({x},{y}) {tile['crop']}: consecutive_unwatered={tile['consecutive_unwatered']}")
        elif tile.get("animal"):
            print(f"({x},{y}) {tile['animal']}: consecutive_unfed={tile['consecutive_unfed']} pending_care_bonus={tile['pending_care_bonus']} fertilizer_available={tile['fertilizer_available']}")

## Debug CARE BONUS — chi5

In [ ]:
for step in range(len(env.steps)):
    obs = env.steps[step][0]["observation"]
    farm = obs["farms"][obs["player"]]
    tile = farm["tiles"][3][3]

    if not isinstance(tile, dict) or tile.get("animal") != "GOOSE":
        continue

    print(f"step={step} day={obs['day']} hour={obs['hour']} yield={tile['yield_units']} fed={tile['fed_today']} care={tile['cared_today']} unfed={tile['consecutive_unfed']} bonus={tile['pending_care_bonus']} fertilizer={tile['fertilizer_available']}")

In [ ]:
for step in range(len(env.steps)):
    obs = env.steps[step][0]["observation"]

    if obs["hour"] != 0:
        continue

    farm = obs["farms"][obs["player"]]
    tile = farm["tiles"][3][3]

    if not isinstance(tile, dict) or tile.get("animal") != "GOOSE":
        continue

    print(f"day={obs['day']} yield={tile['yield_units']} unfed={tile['consecutive_unfed']} bonus={tile['pending_care_bonus']} fertilizer={tile['fertilizer_available']}")

In [ ]:
for step in range(len(env.steps)):
    obs = env.steps[step][0]["observation"]

    if obs["hour"] != 0:
        continue

    farm = obs["farms"][obs["player"]]
    tile = farm["tiles"][4][3]

    if not isinstance(tile, dict) or tile.get("animal") != "SHEEP":
        continue

    print(f"day={obs['day']} yield={tile['yield_units']} unfed={tile['consecutive_unfed']} bonus={tile['pending_care_bonus']} fertilizer={tile['fertilizer_available']}")

In [ ]:
for step in range(len(env.steps)):
    obs = env.steps[step][0]["observation"]

    if obs["hour"] != 0:
        continue

    farm = obs["farms"][obs["player"]]

    for x, y in [(4, 3), (4, 4)]:
        tile = farm["tiles"][y][x]

        if not isinstance(tile, dict) or tile.get("animal") != "COW":
            continue

        print(f"day={obs['day']} pos=({x},{y}) yield={tile['yield_units']} unfed={tile['consecutive_unfed']} bonus={tile['pending_care_bonus']} fertilizer={tile['fertilizer_available']}")

## Debug CARE — chi7


In [ ]:
ANIMAL_MAX_HELD = {"GOOSE": 4, "COW": 6, "SHEEP": 6}

for step in range(len(env.steps)):
    obs = env.steps[step][0]["observation"]
    farm = obs["farms"][obs["player"]]

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):
            if not isinstance(tile, dict) or not tile.get("animal"):
                continue

            animal = tile["animal"]
            yield_units = tile["yield_units"]
            bonus = tile["pending_care_bonus"]
            max_held = ANIMAL_MAX_HELD[animal]
            useful_care = bonus + 1 < max_held

            print(f"step={step} day={obs['day']} hour={obs['hour']} pos=({x},{y}) animal={animal} yield={yield_units} bonus={bonus} max={max_held} cared={tile['cared_today']} useful_care={useful_care}")


## Vérification automatique du plafond de CARE


In [ ]:
ANIMAL_MAX_HELD = {"GOOSE": 4, "COW": 6, "SHEEP": 6}
errors = []

for step in range(len(env.steps) - 1):
    obs = env.steps[step][0]["observation"]
    next_obs = env.steps[step + 1][0]["observation"]
    farm = obs["farms"][obs["player"]]
    next_farm = next_obs["farms"][next_obs["player"]]

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):
            if not isinstance(tile, dict) or not tile.get("animal"):
                continue

            next_tile = next_farm["tiles"][y][x]
            if not isinstance(next_tile, dict) or next_tile.get("animal") != tile["animal"]:
                continue

            max_held = ANIMAL_MAX_HELD[tile["animal"]]
            care_was_useless = tile["pending_care_bonus"] + 1 >= max_held
            care_happened = not tile["cared_today"] and next_tile["cared_today"]

            if care_was_useless and care_happened:
                errors.append((step, obs["day"], obs["hour"], (x, y), tile["animal"], tile["pending_care_bonus"]))

if errors:
    print("CARE inutiles détectés:")
    for error in errors:
        print(error)
else:
    print("OK: aucun CARE effectué lorsque bonus + 1 >= max_held.")
